In [16]:
import numpy as np

np.set_printoptions(suppress=True)

params=np.random.uniform(low=-50, high=150, size=20)#generate random parameters

#set the first three values as the maximum, minimum and zero respectively 
#for better view of effect of quantization on these numbers
params[0]=params.max()+1
params[1]=params.min()-1
params[2]=0

params=np.round(params,2)#round to second decimal

print(params)

[140.22 -47.28   0.   -33.64 139.22  54.59  89.96  97.69  17.28  67.01
  71.85  98.76 109.21 137.86  35.6   30.03 -46.28 130.42  85.92  78.79]


In [17]:
#defining the clamp function which helps in keeping the quantization in bounds
def clamp(params_q: np.array, lower_bound: int, upper_bound: int)->np.array:
    params_q[params_q<lower_bound]=lower_bound
    params_q[params_q>upper_bound]=upper_bound
    return params_q

#now the implementation of asymmetric quantization
def asymmetric_quantization(params: np.array, bits: int) -> tuple[np.array,float,int]:
    #find all values of the variables in the quantization formula
    alpha=np.max(params)
    beta=np.min(params)
    scale=(alpha-beta)/(2**bits-1)
    zero=-1*np.round(beta/scale)
    lower_bound=0
    upper_bound=2**bits-1
    #quantize the parameters
    quantized=clamp(np.round(params/scale + zero),lower_bound,upper_bound).astype(np.int32)
    return quantized, scale, zero

def asymmetric_dequantization(params_q: np.array, scale:float, zero:int)-> np.array:
    return(params_q-zero)*scale



In [18]:
#symmetric quantization
def symmetric_quantization(params: np.array, bits: int) -> tuple[np.array,float]:
    alpha=np.max(np.abs(params))
    scale= alpha/(2**(bits-1)-1)
    lower_bound=-2**(bits-1)
    upper_bound=2**(bits-1)-1
    #quantize the parameters
    quantized=clamp(np.round(params/scale),lower_bound,upper_bound).astype(np.int32)
    return quantized, scale

def symmetric_dequantization(params_q: np.array, scale:float)-> np.array:
    return params_q*scale

In [19]:
#this function is for finding the error or loss of precision after dequantization
def quantization_error(params: np.array, params_q: np.array):
    return np.mean((params-params_q)**2)#this is the MSE(mean squared error)


In [ ]:
(asymmetric_q, asymmetric_scale,asymmetric_zero)=asymmetric_quantization(params,8)
(symmetric_q, symmetric_scale)=symmetric_quantization(params,8)

print(f"Original: ")
print(np.round(params, 2))
print(" ")

print(f"Asymmetric Scale: {asymmetric_scale}, Zero: {asymmetric_zero}")
print(asymmetric_q)

print(f"Symmetric scale: {symmetric_scale}")
print(symmetric_q)


Original: 
[140.22 -47.28   0.   -33.64 139.22  54.59  89.96  97.69  17.28  67.01
  71.85  98.76 109.21 137.86  35.6   30.03 -46.28 130.42  85.92  78.79]
 
Asymmetric Scale: 0.7352941176470589, Zero: 64.0
[255   0  64  18 253 138 186 197  88 155 162 198 213 251 112 105   1 241
 181 171]
Symmetric scale: 1.1040944881889763
[127 -43   0 -30 126  49  81  88  16  61  65  89  99 125  32  27 -42 118
  78  71]


In [22]:
#dequantize back to 32 bits
params_deq_asymmetric=asymmetric_dequantization(asymmetric_q,asymmetric_scale, asymmetric_zero)
params_deq_symmetric=symmetric_dequantization(symmetric_q,symmetric_scale)

print(f"Original: ")
print(np.round(params, 2))
print(" ")

print(f"Dequantize Asymmetric: ")
print(np.round(params_deq_asymmetric,2))
print(" ")

print(f"Dequantized Symmetric: ")
print(np.round(params_deq_symmetric,2))

Original: 
[140.22 -47.28   0.   -33.64 139.22  54.59  89.96  97.69  17.28  67.01
  71.85  98.76 109.21 137.86  35.6   30.03 -46.28 130.42  85.92  78.79]
 
Dequantize Asymmetric: 
[140.44 -47.06   0.   -33.82 138.97  54.41  89.71  97.79  17.65  66.91
  72.06  98.53 109.56 137.5   35.29  30.15 -46.32 130.15  86.03  78.68]
 
Dequantized Symmetric: 
[140.22 -47.48   0.   -33.12 139.12  54.1   89.43  97.16  17.67  67.35
  71.77  98.26 109.31 138.01  35.33  29.81 -46.37 130.28  86.12  78.39]


In [24]:
#calculate the dequantization error
print(f"{"Asymmmetric error: "}{np.round(quantization_error(params, params_deq_asymmetric),2)}")
print(f"{"Symmmetric error: "}{np.round(quantization_error(params, params_deq_symmetric),2)}")

Asymmmetric error: 0.05
Symmmetric error: 0.1
